In [5]:
import numpy as np
from numpy.linalg import eigh
import torch

**A note on the code below:**

When we conduct the diagonalization of $V\Lambda V^*$, we wish to use `eigh` instead of `eig` since `eigh` uses a faster algorithm that **assumes the matrix is Hermitian** $(A = A^*)$. That is to say, we should only pass in Hermitian matrices into `eigh`. This poses a problem since our normal matrix $V\Lambda V^*$ is not Hermitian. Below is what $V\Lambda V^*$ is when $P=3$:

\begin{bmatrix}
-0.5 & 0.866 & 1.118 \\
-0.866 & -0.5 & 1.936 \\
-1.118 & -1.936 & -0.5
\end{bmatrix}

However, $K = V\Lambda V^* - \frac{1}{2}I$ is *skew-Hermitian* $(-A = A^*)$. This means $V\Lambda V^*$ is of the form:
$$V\Lambda V^* = \alpha I + K$$
Where $\alpha\in\mathbb{R}$ and $K$ is skew-Hermitian. Now, observe that:
$$V\Lambda V^*v = \lambda v \iff Kv = (\lambda - \alpha)v$$
That is to say that $V\Lambda V^*$ and its skew part $K$ **share the same eigenvectors**, albeit their eigenvalues are shifted by $\alpha$.

If we multiply both sides by $-i$:
$$-iV\Lambda V^* = -i\alpha I + -iK$$
We then have:
$$(-iV\Lambda V^*)v = \mu v \iff (-iK)v = (\mu + i\alpha)v$$
Which is to say the **eigenvectors of $V\Lambda V^*$, $K$, $-iV\Lambda V^*$, and $-iK$ are all the same!** Their eigenvalues are simply scaled $(\mu = -i\lambda)$. Lucky for us, $-iK$ is Hermitian, so we will seek to (indirectly) use it as the input for `eigh`.

To see why $-iK$ is Hermitian, first note that it is a fact that multiplying a skew-Hermitian matrix by $-i$ yields a Hermitian matrix:
$$\text{Sketch proof: } -K=K^* \implies -iK = i(-K) = iK^* = (-iK)^*$$

<!-- So we have the following:
$$-iV\Lambda V^* = \underbrace{-i\alpha I}_{\text{skew-Hermitian}} + \underbrace{-iK}_{\text{Hermitian}} $$ -->

What `eigh` really does is "take the lower triangular part of the matrix and assumes that the upper triangular part is defined by the symmetry of the matrix" [(stackoverflow)](https://stackoverflow.com/questions/45434989/numpy-difference-between-linalg-eig-and-linalg-eigh). The [documentation](https://numpy.org/doc/2.3/reference/generated/numpy.linalg.eigh.html) for `eigh` actually states in regards to the optional second argument `UPLO`:
> Specifies whether the calculation is done with the lower triangular part of a (‘L’, default) or the upper triangular part (‘U’). Irrespective of this value only the real parts of the diagonal will be considered in the computation to preserve the notion of a Hermitian matrix. It therefore follows that the imaginary part of the diagonal will always be treated as zero.

Here we note that $-iV\Lambda V^*$ has a *purely imaginary diagonal*, so if we pass it into `eigh`, it will effectively be treated as $-iK$ since $K$ is just $V\Lambda V^*$ with the diagonal removed. So we leverage this technical nuance to avoid computing $K$ and just pass in `VΛV * -1j` to yield the desired eigenvector matrix $V$.




In [6]:
def DPLR_HiPPO(P):
    
    # HiPPO
    P_arr = np.arange(P)
    M = np.sqrt(2*P_arr + 1) # creates a P-dimensional array according to HiPPO-LegS
    A = -np.tril(np.outer(M,M)) + np.diag(P_arr) # A_nk in Appendix C.1 of S4 paper
    
    # DPLR
    Q = np.sqrt(P_arr + 0.5) # low-rank array (i.e. vector) part of the "low rank correction". See "adding..." in C.1 of S4 paper
    VΛV = A + np.outer(Q,Q)
    Λ_real = np.diagonal(VΛV) # this is just a P-dimensional array of -0.5's
    Λ_imag, V = eigh(VΛV * -1j)
    Q_tilde = V.conj() @ Q
    
    B = np.sqrt(2 * P_arr + 1.0)
    B_tilde = V.conj() @ B   
    
    return (
        # note the imaginary eigenvalues are being multiplied by i to account for the -i scaling above
        # recall that \mu = -i\lambda, so to recover \lambda we just multiply by i
        torch.tensor(np.asarray(Λ_real + 1j * Λ_imag), dtype=torch.complex64),
        torch.tensor(np.asarray(V), dtype=torch.complex64),
        torch.tensor(np.asarray(Q_tilde)),
        torch.tensor(np.asarray(B_tilde))
    ) 
    

If you wish to have confirmation that this diagonalization works, run the cell below for any value of `P` you desire.

In [7]:
P = 3
P_arr = np.arange(P)
M = np.sqrt(2*P_arr + 1) # creates a P-dimensional array according to HiPPO-LegS
A = -np.tril(np.outer(M,M)) + np.diag(P_arr) # A_nk in Appendix C.1 of S4 paper

# NPLR
Q = np.sqrt(P_arr + 0.5) # part of the "low rank correction".
VΛV = A + np.outer(Q,Q)
Λ_real = np.diagonal(VΛV) # this is just a P-dimensional array of -0.5's
Λ_imag, V = eigh(VΛV * -1j)

Λ = np.diag(Λ_real + 1j*Λ_imag)
print(np.allclose(A,V @ Λ @ V.conj().T - np.outer(Q,Q), atol=1e-4, rtol=1e-4))


True


Now we set up the forward ZOH variant of the LLH SSM layer.

In [8]:
from typing import Optional, Tuple
import torch.nn as nn
import torch.nn.functional as F

A lot of technical notes for the `Forward_LLH` class below:

#### `_init_A()`:
**Ensuring $\mathfrak{R}(\Lambda)<0$:**

Eventually we will need to discretize $A$ as $\bar{A}=e^{\Lambda\Delta t}$. This raises a risk of numerical instability if the real parts of $\Lambda$ are not negative. Of course, gradient descent is unable to "ignore" negative values during the backpropagation process, so instead a transformation is performed to ensure that only negative eigenvalues are considered.

Instead of making `Λ_P.real` a parameter itself, we represent it via:
$$\theta = \log(-\mathfrak{R}(\Lambda))$$
Note here that $\Lambda$ is initalized as a diagonal matrix whose real parts are all $-0.5$, so logarithm above is well-defined. Gradient descent is performed on $\theta\in\mathbb{R}$, and once $\Lambda$ must be re-constructed, we simply apply:
$$\mathfrak{R}(\Lambda) = -e^{\theta}$$
Hence, we are guaranteed negative real parts for $\Lambda$

**Input-dependent dynamics and `Δ_net`:**

`relative_time` has to do with whether we want to rescale $\Lambda$ at each event time based on the current input $u_{t_i}$ (à la [Mamba](https://openreview.net/forum?id=tEYskw1VY2)). This is the "input-dependent dynamics" section of the paper (see B.3 of S2P2 paper.) $\Delta_i$ is a time-scaling diagonal matrix that changes the relative timescale of each channel of the state's influence (a scaling for each of the $P$ channels). Large values suggest time should pass quickly for that channel, suggesting faster convergence to steady-state, and smaller values will cause the model to retain the influence that prior events have on future ones due to the slower perceived passage of time. If `relative_time = True`, we make this a learnable network via $$\Delta_i = \text{diag(Softplus}(W * u_{t_i} + b))$$ and call it `Δ_net`. It is then multiplied by $\Lambda$ to get a rescaled diagonal matrix $\Lambda_i$

**Initializing the bias of $\Delta_i$ in a numerically sounds way:**

Note that $$\text{Softplus}(x) = \log(1 + e^x)$$ Recall that we are multiplying $\Delta_i=\text{diag(Softplus}(W * u_{t_i} + b))$ by $\Lambda$, so at initialization we want the softplus's output to be close to 1, so $\Lambda_i$ is close to $\Lambda$. Now, we already have that the input $u$ at initialization is 0 (in the first layer), so the $W*u$ term in the
softplus will be 0. so we are left with initializing the bias such that $\text{Softplus}(b) = 1$
We can show what this looks like algebraically:
$$\text{Softplus}(b) = \log(1 + e^b) = 1 \rightarrow b = \log(e^1 - 1)$$
The bias vector is initialized as a $P$-dimensional tensor of ones, then add `log(-expm1(-1))`$=\log(-(e^{-1}-1))$ to each element,
which is algebraically equivalent to the $\log(e^1 - 1)$ desired.

Why is this done? Because the expression $e^x - 1$ is numerically unstable (e.g. catastrophic cancellation) for small values of $x$. $x + log(1-e^{-x})$ is more numerically stable for small values of x than $\log(e^x - 1)$.

#### `_init_B()`:

Although there is a theoretical HiPPO B, it was found in [Gu et al., 2022](https://papers.neurips.cc/paper_files/paper/2022/file/e9a32fade47b906de908431991440f7c-Paper-Conference.pdf) on the beginning of page 6 that it was not necessary to initialize $B$ according to the HiPPO, only $A$, due to its dominant effect on the dynamics of the SSM. However, allowing $B$ to be learned freely still allows for some material gain, so they opt to initialize $B$ randomly with Xavier normal initialization (while still multiplying it by the eigenvector matrix $V$.)

#### `forward()`:

**Broadcasting the initial state `x_0_P` across all the batches:**

The initial state tensor `x_0_P` is only $P$-dimensional, but we need it to be broacast across all the batches as a $B\times P$-dimensional tensor. This is accomplished by appending dimensions via `view()` and setting those dimensions via `expand()`.

The `view()` method returns a new tensor with the same data but of a different shape, without copying memory (see [documentation](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.view.html) and [stackoverflow](https://stackoverflow.com/questions/42479902/what-does-view-do-in-pytorch).)

The `*` operator is known as the "packing/unpacking" operator. The `view()` method takes arguments one number at a time, not in a list, so since we want to feed the numbers from the list into `view()` we unpack it with `*` (i.e. `foo(*[1,2,3])` is equivalent to `foo(1,2,3)`)

`expand()` does the final broadcasting without copying data (see [documentation](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.expand.html)). So if the current tensor has shape `(1,1,64)`, if we call `.expand(32,100,-1)`, it produces a tensor of size `(32,100,64)` the `-1` denotes "keep this dimension the original size", whereas the "1" dimensions are broadcasted to the indicated sizes.

**Checking the dimensions between input signal and mark impulse:**

The `assert all()` block found is just a shape check between the input signal and mark impulse. the `zip()` method pairs elements of one list with the elements of the other list in order. so if `x.shape = y.shape = (B,N,H)`, then `zip(x.shape,y.shape)` will produce pairs `(B,B)` `(N,N)` and `(H,H)` the `for` loop checks to see that each of these dimensions match, and the `assert all()` checks that every output of the `for` loop is True. (`all()` returns True if every element is True, False otherwise) `assert` will throw an error if it returns false.

**`pre_norm` vs. `post_norm`:**

Most models in the current literature discuss and analyze the implications of applying $\text{LayerNorm}$ at the end of the current pass, or the beginning of the next pass. This has material implications in the sense that in the post-normalization approach:
$$u^{(l+1)} = \text{LayerNorm}(\sigma(y^{(l)}) + u^{(l)})$$
the normalization is applied *after* adding the residual stream $u^{(l)}$. This means the distribution of the added residual will also be shifted. When compared to the pre-normalization approach:
$$u^{(l+1)} = \text{LayerNorm}(\sigma(y^{(l)})) + u^{(l)}$$
The residual stream's distribution (across the $H$ features) is not considered.

In discussion with the one of the model's authors, they stated
>Empirically, we found post-norm works well in our model, likely because the depth of our architecture remains within a range where the benefits of better signal preservation outweigh the optimization instabilities typically seen in much deeper models.



In [9]:
class Forward_LLH(nn.Module):
    
    def __init__(
        self,
        P: int,
        H: int,
        dropout_rate: float = 0.0,
        pre_norm: bool = True,
        post_norm: bool = False,
        is_first_layer: bool = False,
        relative_time: bool = False
    ):
        super(Forward_LLH, self).__init__()
        
        # inscribe config file parameters as class attributes
        self.P = P
        self.H = H
        self.dropout_rate = dropout_rate
        self.pre_norm = pre_norm
        self.post_norm = post_norm
        self.is_first_layer = is_first_layer
        self.relative_time = relative_time
        
        self.act_func = nn.Sequential(nn.GELU(), nn.Dropout(p=self.dropout_rate)) # D(GELU())
        self.norm = nn.LayerNorm(self.H) # we will normalize over the H features of the input and output
        
        # the multiplication by .001 is in line with the practice of avoiding uncontrolled intial state magnitude that can 
        # destabilize training. 
        self.x_0_P = nn.Parameter(torch.complex(torch.randn(self.P), torch.randn(self.P)) * 1e-3, 
                                  requires_grad=True)
        
        self._init_ssm_params()
        
    def _init_ssm_params(self):
        self._init_A()
        if not self.is_first_layer:
            self._init_B()
        self._init_C()
        if not self.is_first_layer:
            self._init_D()
        self._init.E()
        
    def _init_A(self):        
        Λ_P, V_PP, _, _ = DPLR_HiPPO(self.P)
        
        self.V_PP = V_PP
        self.V_star_PP = V_PP.conj().T
        
        # the real part of Λ is initialized to -0.5 across all P channels so it is guaranteed that log is receiving 
        # a positive input here
        self.log_neg_real_Λ_P = nn.Parameter((-Λ_P.real).log()) 
        self.imag_Λ_P = nn.Parameter(Λ_P.imag)
        
        if self.relative_time:
            self.Δ_net = nn.Linear(self.H, self.P, bias=True)
            with torch.no_grad():
                self.Δ_net.weight.copy_(nn.init.xavier_normal_(self.Δ_net.weight))
                
                bias = torch.ones(self.P)
                bias += torch.log(-torch.expm1(-bias))
                self.Δ_net.bias.copy_(bias)
        else:
            self.log_step_size_P = nn.Parameter(torch.zeros(size=(self.P)), requires_grad=False) # this may be unneeded
            
    @property
    def Λ_P(self):
        return torch.complex(-torch.exp(self.log_neg_real_Λ_P), self.imag_Λ_P)
    
    def _init_B(self):
        B = nn.init.xavier_normal_(torch.zeros((self.P, self.H)))
        B_tilde_PH = self.V_star_PP @ B.type(torch.complex64) # TODO: paper says we multiply by negative V*?
        self.B_tilde_PH = nn.Parameter(B_tilde_PH)
        
    def _init_C(self):       
        C = nn.init.xavier_normal_(torch.zeros((self.H, self.P)))
        C_tilde_HP = C.type(torch.complex64) @ self.V_PP
        self.C_tilde_HP = nn.Parameter(C_tilde_HP)
        
    def _init_D(self):        
        D_H = torch.zeros(self.H)
        nn.init.normal_(D_H, std=1.0)
        self.D_H = nn.Parameter(D_H)
        
    def _init_E(self):        
        E = nn.init.xavier_normal_(torch.zeros((self.P, self.H))) # R = H
        E_tilde_PH = self.V_star_PP @ E.type(torch.complex64)
        self.E_tilde_PH = nn.Parameter(E_tilde_PH)
        
    def compute_impulse(self,α_H):
        # although the paper defines α as a RxK (we set R = H) matrix, we are only taking a column vector from this matrix
        tilde_Eα_P = torch.einsum("ph,...h->...p", self.E_tilde_PH, α_H.type(torch.complex64))
        return tilde_Eα_P
    
    def get_Λ_i(self, right_u_NH, shift_u=True):
        if self.relative_time and (right_u_NH is not None): # right_u is null for first layer
            if shift_u:
                right_u_NH = F.pad(right_u_NH[..., :-1,:], (0,0,1,0)) # TODO: maybe explain this
            Λ_i_NP = F.softplus(self.Δ_net(right_u_NH)) * self.Λ_P
            return {"Λ_i_NP": Λ_i_NP} # these keys will be used in _ssm in order to determine the dimension
        else:
            if self.relative_time: # relative_time is true but it's the first layer (right_u is null)
                Λ_i_P = F.softplus(self.Δ_net.bias) * self.Λ_P # there is no u to multiply the weight matrix by
            else:
                Λ_i_P = self.Λ_P
            return {"Λ_i_P": Λ_i_P} # these keys will be used in _ssm in order to determine the dimension
        
    def forward(
        self,
        left_u_NH: Optional[torch.Tensor],
        right_u_NH: Optional[torch.Tensor],
        α_NH: torch.Tensor,
        dt_N: torch.Tensor, # [0, t1-t0, ..., t_N-t_{N-1}]
        x_0_P: Optional[torch.Tensor] = None # this argument may be omitted
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        
        *leading_dims, _, _ = α_NH.shape # leading_dims will typically just be batch size B
        num_leading_dims = len(leading_dims)
        
        # the state is P-dimensional, but it needs to be broacast across the B sequences 
        # basically we just need to reformat initial_state_P as a [...,P] tensor (most likely a [B,P] tensor)
        if x_0_P is None:
            x_0_P = self.x_0_P.view(*[1 for _ in range(num_leading_dims)], -1).expand(*leading_dims, -1)
        normed_left_u_NH = left_u_NH
        normed_right_u_NH = right_u_NH
        if normed_left_u_NH is not None:
            # checking that all the dimensions match
            assert all(u_d == a_d for u_d, a_d in zip(normed_left_u_NH.shape, α_NH.shape))
            if self.pre_norm:
                normed_left_u_NH = self.norm(normed_left_u_NH) # normed_left_u_NH is not actually norm'd until now
        if normed_right_u_NH is not None:
            # checking that all the dimensions match
            assert all(u_d == a_d for u_d, a_d in zip(normed_right_u_NH.shape, α_NH.shape))
            if self.pre_norm:
                normed_right_u_NH = self.norm(normed_right_u_NH) # normed_right_u_NH is not actually norm'd until now
        
        # this _ssm() call is what actually carries out the bulk of of Algorithm 1
        right_x_NP, left_y_NH, right_y_NH = self._ssm(
            left_u_NH = normed_left_u_NH,
            right_u_NH = normed_right_u_NH,
            tilde_Eα_NP = self.compute_impulse(α_NH),
            dt_N = dt_N,
            x_0_P = x_0_P
        )
        
        next_layer_left_u_NH = next_layer_right_u_NH = None
        if left_y_NH is not None:
            next_layer_left_u_NH = self.act_func(left_y_NH) + (left_u_NH if left_u_NH is not None else 0.0)
            if self.post_norm:
                next_layer_left_u_NH = self.norm(next_layer_left_u_NH)
        if right_y_NH is not None:
            next_layer_right_u_NH = self.act_func(right_y_NH) + (right_u_NH if right_u_NH is not None else 0.0)
            if self.post_norm:
                next_layer_right_u_NH = self.norm(next_layer_right_u_NH)
        
        return right_x_NP, next_layer_left_u_NH, next_layer_right_u_NH
    
    def _ssm(
        self,
        left_u_NH: Optional[torch.Tensor],
        right_u_NH: Optional[torch.Tensor],
        tilde_Eα_NP: torch.Tensor,
        dt_N: torch.Tensor,
        x_0_P: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        
        *leading_dims, N, P = tilde_Eα_NP.shape
        
        Λ_i = self.get_Λ_i(right_u_NH, shift_u=True)
        if "Λ_i_P" in Λ_i: # we use this key check instead of shape check because both cases end in the dimension of P
            # TODO: check why there is no ellipsis in second argument of einsum. no batch?
            Λ_dt_NP = torch.einsum("...n,p->...np", dt_N, Λ_i["Λ_i_P"]) # element-wise product but a diagonal matrix
        else:
            # scaling each row of Λ_i_NP by associated element of dt_N
            Λ_dt_NP = torch.einsum("...n,...np->...np", dt_N, Λ_i["Λ_i_NP"]) 
        
        # compute left-hand Du    
        if left_u_NH is not None:
            left_Du_NH = torch.einsum("...nh,h->...nh", left_u_NH, self.D_H)
        else:
            assert self.is_first_layer
            left_Du_NH = 0.0
            
        # computing right-hand Bu and Du
        if right_u_NH is not None:
            right_u_NH = F.pad(right_u_NH[..., :-1, :], (0, 0, 1, 0))
            right_Bu_NP = torch.einsum( # Equation 19 of S2P2 paper
                "...np,ph,...nh->...np",
                Λ_dt_NP.exp() - 1, # Λ_bar - 1
                self.B_tilde_PH,
                right_u_NH.type(torch.complex64)
            )
            right_Du_NH = torch.einsum("...nh,h->...nh", right_u_NH, self.D_H)
        else:
            assert self.is_first_layer
            right_Du_NH = right_Bu_NP =  0.0
            
        # parallel scan as per (Heinsen 2023) and https://github.com/PeaBrane/mamba-tiny/blob/master/scans.py
        log_impulse_Np1_P = torch.concat((x_0_P.unsqueeze(-2), right_Bu_NP + tilde_Eα_NP), dim = -2).log()
        Λ_dt_star = F.pad(Λ_dt_NP.cumsum(-2), (0, 0, 1, 0))
        right_log_x_NP = torch.logcumsumexp(log_impulse_Np1_P - Λ_dt_star, -2) + Λ_dt_star
        right_x_NP = right_log_x_NP.exp() # Contains initial_state_P in index 0
        left_x_NP = Λ_dt_NP.exp() * right_x_NP[..., :-1, :] + right_Bu_NP # this is Equation (15)
        right_x_NP = right_x_NP[..., 1:, :] # finally we exponentiate, ignoring the first dimension padded earlier
        
        left_y_NH = 2 * torch.einsum("hp,...np->...nh", self.C_tilde_HP, left_x_NP).real + left_Du_NH
        right_y_NH = 2 * torch.einsum("hp,...np->...nh", self.C_tilde_HP, right_x_NP).real + right_Du_NH
        
        return right_x_NP, left_y_NH, right_y_NH
        
    def get_x_left_limit(
        self,
        right_x_P: torch.Tensor,
        dt_G: torch.Tensor,
        current_right_u_H: torch.Tensor
    ) -> torch.Tensor:
        
        if current_right_u_H is not None and self.pre_norm:
            current_right_u_H = self.norm(current_right_u_H)
            
        Λ_i = self.get_Λ_i(current_right_u_H, shift_u = False) # input signal u should already been shifted
        if "Λ_i_P" in Λ_i:
            Λ_bar_GP = torch.exp(torch.einsum("...g,p->...gp", dt_G, Λ_i["Λ_i_P"]))
        else:
            Λ_bar_GP = torch.exp(torch.einsum("...g,...p->...gp", dt_G, Λ_i["Λ_i_NP"]))
            
        Λ_bar_x_GP = torch.einsum("...p,...gp->...gp", right_x_P, Λ_bar_GP)
        
        if current_right_u_H is None:
            assert self.is_first_layer
            return Λ_bar_x_GP
        else: # add Bu to impulse
            if self.pre_norm:
                current_right_u_H = self.norm(current_right_u_H)
            impulse_GP = torch.einsum(
                "...gp,ph,...h->...gp", 
                Λ_bar_GP - 1.0, 
                self.B_tilde_PH, 
                current_right_u_H.type(torch.complex64))
            return Λ_bar_x_GP + impulse_GP
    
    def depth_pass(
        self,
        current_left_x_P: torch.Tensor,
        current_left_u_H: Optional[torch.Tensor]
    ) -> torch.Tensor:
        
        if current_left_u_H is not None:
            if self.pre_norm:
                normed_u_H = self.norm(current_left_u_H)
            else:
                normed_u_H = current_left_u_H
            left_Du_H = torch.einsum("...h,h->...h", normed_u_H, self.D_H)
        else:
            assert self.is_first_layer
            left_Du_H = 0.0
            
        y_H = 2 * torch.einsum("...p,hp->...h", current_left_x_P, self.C_tilde_HP).real + left_Du_H
        
        if self.post_norm:
            new_u_H = self.norm(self.act_func(y_H) + (current_left_u_H if current_left_u_H is not None else 0.0))
        else:
            new_u_H = self.act_func(y_H) + (current_left_u_H if current_left_u_H is not None else 0.0)
            
        return new_u_H
        

### Some base classes

In [10]:
from typing import List, Optional, Tuple, Union
import torch
from torch import nn
import math
from LLH import Forward_LLH

In [11]:
# Embedding layer for complex-valued embeddings for the diagonalized SSM matrices
class ComplexEmbedding(nn.Module):
    def __init__(self, *args, **kwargs):
        super(ComplexEmbedding, self).__init__()
        self.real_embedding = nn.Embedding(*args, **kwargs)
        self.imag_embedding = nn.Embedding(*args, **kwargs)
        
        self.real_embedding.weight.data *= 1e-3
        self.imag_embedding.weight.data *= 1e-3
        
    def forward(self, x):
        return torch.complex(self.real_embedding(x), self.imag_embedding(x))
    
class ScaledSoftPlus(nn.Module):
    def __init__(self, num_marks, threshold=20.):
        super(ScaledSoftPlus, self).__init__()
        self.threshold = threshold
        self.log_beta = nn.Parameter(torch.zeros(num_marks), requires_grad=True)
        
    def forward(self, x):
        beta = self.log_beta.exp()
        beta_x = beta * x
        return torch.where( # if above threshold, then the transform is effectively linear
            beta_x <= self.threshold, 
            torch.log1p(beta_x.clamp(max=math.log(1e5)).exp()) / beta, 
            x
        )
        
class IntensityNet(nn.Module):
    def __init__(self, input_dim, num_event_types, bias_bool):
        super().__init__()
        self.intensity_net = nn.Linear(input_dim, num_event_types, bias=bias_bool)
        self.softplus = ScaledSoftPlus(num_event_types)
        
    def forward(self, x):
        return self.softplus(self.intensity_net(x))

### Utilities

In [12]:

def is_torch_mps_available():
    try:
        import torch
        torch.device('mps')
        return True
    except RuntimeError:
        return False

def set_device(gpu=-1):
    """Setup the device.

    Args:
        gpu (int, optional): num of GPU to use. Defaults to -1 (not use GPU, i.e., use CPU).
    """
    if gpu >= 0:
        if torch.cuda.is_available():
            device = torch.device("cuda:" + str(gpu))
        elif is_torch_mps_available():
            device = torch.device("mps")
    else:
        device = torch.device("cpu")
    return device

#### Event Sampler

In [13]:
import torch
import torch.nn as nn
# from easy_tpp.utils import logger


class EventSampler(nn.Module):
    """Event Sequence Sampler based on thinning algorithm, which corresponds to Algorithm 2 of
    The Neural Hawkes Process: A Neurally Self-Modulating Multivariate Point Process,
    https://arxiv.org/abs/1612.09328.

    The implementation uses code from https://github.com/yangalan123/anhp-andtt/blob/master/anhp/esm/thinning.py.
    """

    def __init__(self, num_sample, num_exp, over_sample_rate, num_samples_boundary, dtime_max, patience_counter,
                 device):
        """Initialize the event sampler.

        Args:
            num_sample (int): number of sampled next event times via thinning algo for computing predictions.
            num_exp (int): number of i.i.d. Exp(intensity_bound) draws at one time in thinning algorithm
            over_sample_rate (float): multiplier for the intensity up bound.
            num_samples_boundary (int): number of sampled event times to compute the boundary of the intensity.
            dtime_max (float): max value of delta times in sampling
            patience_counter (int): the maximum iteration used in adaptive thinning.
            device (torch.device): torch device index to select.
        """
        super(EventSampler, self).__init__()
        self.num_sample = num_sample
        self.num_exp = num_exp
        self.over_sample_rate = over_sample_rate
        self.num_samples_boundary = num_samples_boundary
        self.dtime_max = dtime_max
        self.patience_counter = patience_counter
        self.device = device

    def compute_intensity_upper_bound(self, time_seq, time_delta_seq, event_seq, intensity_fn,
                                      compute_last_step_only):
        # logger.critical(f'time_seq: {time_seq}')
        # logger.critical(f'time_delta_seq: {time_delta_seq}')
        # logger.critical(f'event_seq: {event_seq}')
        # logger.critical(f'intensity_fn: {intensity_fn}')
        # logger.critical(f'compute_last_step_only: {compute_last_step_only}')
        """Compute the upper bound of intensity at each event timestamp.

        Args:
            time_seq (tensor): [batch_size, seq_len], timestamp seqs.
            time_delta_seq (tensor): [batch_size, seq_len], time delta seqs.
            event_seq (tensor): [batch_size, seq_len], event type seqs.
            intensity_fn (fn): a function that computes the intensity.
            compute_last_step_only (bool): wheter to compute the last time step pnly.

        Returns:
            tensor: [batch_size, seq_len]
        """
        batch_size, seq_len = time_seq.size()

        # [1, 1, num_samples_boundary]
        time_for_bound_sampled = torch.linspace(start=0.0,
                                                end=1.0,
                                                steps=self.num_samples_boundary,
                                                device=self.device)[None, None, :]

        # [batch_size, seq_len, num_samples_boundary]
        dtime_for_bound_sampled = time_delta_seq[:, :, None] * time_for_bound_sampled

        # [batch_size, seq_len, num_samples_boundary, event_num]
        intensities_for_bound = intensity_fn(time_seq,
                                             time_delta_seq,
                                             event_seq,
                                             dtime_for_bound_sampled,
                                             max_steps=seq_len,
                                             compute_last_step_only=compute_last_step_only)

        # [batch_size, seq_len]
        bounds = intensities_for_bound.sum(dim=-1).max(dim=-1)[0] * self.over_sample_rate

        return bounds

    def sample_exp_distribution(self, sample_rate):
        """Sample an exponential distribution.

        Args:
            sample_rate (tensor): [batch_size, seq_len], intensity rate.

        Returns:
            tensor: [batch_size, seq_len, num_exp], exp numbers at each event timestamp.
        """

        batch_size, seq_len = sample_rate.size()

        # For fast approximation, we reuse the rnd for all samples
        # [batch_size, seq_len, num_exp]
        exp_numbers = torch.empty(size=[batch_size, seq_len, self.num_exp],
                                  dtype=torch.float32,
                                  device=self.device)

        # [batch_size, seq_len, num_exp]
        # exp_numbers.exponential_(1.0)
        exp_numbers.exponential_(1.0)

        # [batch_size, seq_len, num_exp]
        # exp_numbers = torch.tile(exp_numbers, [1, 1, self.num_sample, 1])

        # [batch_size, seq_len, num_exp]
        # div by sample_rate is equivalent to exp(sample_rate),
        # see https://en.wikipedia.org/wiki/Exponential_distribution
        exp_numbers = exp_numbers / sample_rate[:, :, None]

        return exp_numbers

    def sample_uniform_distribution(self, intensity_upper_bound):
        """Sample an uniform distribution

        Args:
            intensity_upper_bound (tensor): upper bound intensity computed in the previous step.

        Returns:
            tensor: [batch_size, seq_len, num_sample, num_exp]
        """
        batch_size, seq_len = intensity_upper_bound.size()

        unif_numbers = torch.empty(size=[batch_size, seq_len, self.num_sample, self.num_exp],
                                   dtype=torch.float32,
                                   device=self.device)
        unif_numbers.uniform_(0.0, 1.0)

        return unif_numbers

    def sample_accept(self, unif_numbers, sample_rate, total_intensities, exp_numbers):
        """Do the sample-accept process.

        For the accumulated exp (delta) samples drawn for each event timestamp, find (from left to right) the first
        that makes the criterion < 1 and accept it as the sampled next-event time. If all exp samples are rejected 
        (criterion >= 1), then we set the sampled next-event time dtime_max.

        Args:
            unif_numbers (tensor): [batch_size, max_len, num_sample, num_exp], sampled uniform random number.
            sample_rate (tensor): [batch_size, max_len], sample rate (intensity).
            total_intensities (tensor): [batch_size, seq_len, num_sample, num_exp]
            exp_numbers (tensor): [batch_size, seq_len, num_sample, num_exp]: sampled exp numbers (delta in Algorithm 2).

        Returns:
            result (tensor): [batch_size, seq_len, num_sample], sampled next-event times.
        """

        # [batch_size, max_len, num_sample, num_exp]
        criterion = unif_numbers * sample_rate[:, :, None, None] / total_intensities
        
        # [batch_size, max_len, num_sample, num_exp]
        masked_crit_less_than_1 = torch.where(criterion<1,1,0)
        
        # [batch_size, max_len, num_sample]
        non_accepted_filter = (1-masked_crit_less_than_1).all(dim=3)
        
        # [batch_size, max_len, num_sample]
        first_accepted_indexer = masked_crit_less_than_1.argmax(dim=3)
        
        # [batch_size, max_len, num_sample,1]
        # indexer must be unsqueezed to 4D to match the number of dimensions of exp_numbers
        result_non_accepted_unfiltered = torch.gather(exp_numbers, 3, first_accepted_indexer.unsqueeze(3))
        
        # [batch_size, max_len, num_sample,1]
        result = torch.where(non_accepted_filter.unsqueeze(3), torch.tensor(self.dtime_max), result_non_accepted_unfiltered)
        
        # [batch_size, max_len, num_sample]
        result = result.squeeze(dim=-1)
        
        return result

    def draw_next_time_one_step(self, time_seq, time_delta_seq, event_seq, dtime_boundary,
                                intensity_fn, compute_last_step_only=False):
        """Compute next event time based on Thinning algorithm.

        Args:
            time_seq (tensor): [batch_size, seq_len], timestamp seqs.
            time_delta_seq (tensor): [batch_size, seq_len], time delta seqs.
            event_seq (tensor): [batch_size, seq_len], event type seqs.
            dtime_boundary (tensor): [batch_size, seq_len], dtime upper bound.
            intensity_fn (fn): a function to compute the intensity.
            compute_last_step_only (bool, optional): whether to compute last event timestep only. Defaults to False.

        Returns:
            tuple: next event time prediction and weight.
        """
        # 1. compute the upper bound of the intensity at each timestamp
        # the last event has no label (no next event), so we drop it
        # [batch_size, seq_len=max_len - 1]
        intensity_upper_bound = self.compute_intensity_upper_bound(time_seq,
                                                                   time_delta_seq,
                                                                   event_seq,
                                                                   intensity_fn,
                                                                   compute_last_step_only)

        # 2. draw exp distribution with intensity = intensity_upper_bound
        # we apply fast approximation, i.e., re-use exp sample times for computation
        # [batch_size, seq_len, num_exp]
        exp_numbers = self.sample_exp_distribution(intensity_upper_bound)
        exp_numbers = torch.cumsum(exp_numbers, dim=-1)
        
        # 3. compute intensity at sampled times from exp distribution
        # [batch_size, seq_len, num_exp, event_num]
        intensities_at_sampled_times = intensity_fn(time_seq,
                                                    time_delta_seq,
                                                    event_seq,
                                                    exp_numbers,
                                                    max_steps=time_seq.size(1),
                                                    compute_last_step_only=compute_last_step_only)

        # [batch_size, seq_len, num_exp]
        total_intensities = intensities_at_sampled_times.sum(dim=-1)

        # add one dim of num_sample: re-use the intensity for samples for prediction
        # [batch_size, seq_len, num_sample, num_exp]
        total_intensities = torch.tile(total_intensities[:, :, None, :], [1, 1, self.num_sample, 1])
        
        # [batch_size, seq_len, num_sample, num_exp]
        exp_numbers = torch.tile(exp_numbers[:, :, None, :], [1, 1, self.num_sample, 1])
        
        # 4. draw uniform distribution
        # [batch_size, seq_len, num_sample, num_exp]
        unif_numbers = self.sample_uniform_distribution(intensity_upper_bound)

        # 5. find out accepted intensities
        # [batch_size, seq_len, num_sample]
        res = self.sample_accept(unif_numbers, intensity_upper_bound, total_intensities, exp_numbers)

        # [batch_size, seq_len, num_sample]
        weights = torch.ones_like(res)/res.shape[2]
        
        # add a upper bound here in case it explodes, e.g., in ODE models
        return res.clamp(max=1e5), weights


### Base Model

In [14]:
class TorchBaseModel(nn.Module):
    def __init__(self, model_config):
        super(TorchBaseModel, self).__init__()
        self.loss_integral_num_sample_per_step = model_config.loss_integral_num_sample_per_step
        self.hidden_size = model_config.hidden_size
        self.num_event_types = model_config.num_event_types  # does not include [PAD], [BOS], [EOS]
        self.num_event_types_pad = model_config.num_event_types_pad  # include [PAD], [BOS], [EOS]
        self.pad_token_id = model_config.pad_token_id
        self.eps = torch.finfo(torch.float32).eps
        
        self.layer_type_emb = nn.Embedding(
            self.num_event_types_pad,  # have padding
            self.hidden_size,
            padding_idx=self.pad_token_id,
        )

        self.gen_config = model_config.thinning
        self.event_sampler = None
        self.device = set_device(model_config.gpu)
        self.use_mc_samples = model_config.use_mc_samples

        self.to(self.device)

        if self.gen_config:
            self.event_sampler = EventSampler(
                num_sample=self.gen_config.num_sample,
                num_exp=self.gen_config.num_exp,
                over_sample_rate=self.gen_config.over_sample_rate,
                patience_counter=self.gen_config.patience_counter,
                num_samples_boundary=self.gen_config.num_samples_boundary,
                dtime_max=self.gen_config.dtime_max,
                device=self.device,
            )

    @staticmethod
    def generate_model_from_config(model_config):
        """Generate the model in derived class based on model config.

        Args:
            model_config (EasyTPP.ModelConfig): config of model specs.
        """
        model_id = model_config.model_id

        for subclass in TorchBaseModel.__subclasses__():
            if subclass.__name__ == model_id:
                return subclass(model_config)

        raise RuntimeError("No model named " + model_id)

    @staticmethod
    def get_logits_at_last_step(logits, batch_non_pad_mask, sample_len=None):
        """Retrieve the hidden states of last non-pad events.

        Args:
            logits (tensor): [batch_size, seq_len, hidden_dim], a sequence of logits
            batch_non_pad_mask (tensor): [batch_size, seq_len], a sequence of masks
            sample_len (tensor): default None, use batch_non_pad_mask to find out the last non-mask position

        ref: https://medium.com/analytics-vidhya/understanding-indexing-with-pytorch-gather-33717a84ebc4

        Returns:
            tensor: retrieve the logits of EOS event
        """

        seq_len = batch_non_pad_mask.sum(dim=1)
        select_index = seq_len - 1 if sample_len is None else seq_len - 1 - sample_len
        # [batch_size, hidden_dim]
        select_index = select_index.unsqueeze(1).repeat(1, logits.size(-1))
        # [batch_size, 1, hidden_dim]
        select_index = select_index.unsqueeze(1)
        # [batch_size, hidden_dim]
        last_logits = torch.gather(logits, dim=1, index=select_index).squeeze(1)
        return last_logits
    
    # Generate the time point samples for every interval. Used for integral in log-likelihood
    def make_dtime_loss_samples(self, time_delta_seq): # torch_basemodel
        """Generate the time point samples for every interval.

        Args:
            time_delta_seq (tensor): [batch_size, seq_len].

        Returns:
            tensor: [batch_size, seq_len, n_samples]
        """
        if self.use_mc_samples:
            # [B, N, n_samples]
            dtimes_ratio_sampled = torch.rand(
                (*time_delta_seq.shape, self.loss_integral_num_sample_per_step),
                device=self.device,
            )
        else:
            # [1, 1, n_samples]
            dtimes_ratio_sampled = torch.linspace(
                start=0.0,
                end=1.0,
                steps=self.loss_integral_num_sample_per_step,
                device=self.device,
            )[None, None, :]

        # [batch_size, max_len, n_samples]
        sampled_dtimes = time_delta_seq[:, :, None] * dtimes_ratio_sampled

        return sampled_dtimes
    
    def compute_loglikelihood(self, time_delta_seq, lambda_at_event, lambdas_loss_samples, seq_mask, type_seq):
        """Compute the loglikelihood of the event sequence based on Equation (8) of NHP paper.

        Args:
            time_delta_seq (tensor): [batch_size, seq_len], time_delta_seq from model input.
            lambda_at_event (tensor): [batch_size, seq_len, num_event_types], unmasked intensity at
            (right after) the event.
            lambdas_loss_samples (tensor): [batch_size, seq_len, num_sample, num_event_types],
            intensity at sampling times.
            seq_mask (tensor): [batch_size, seq_len], sequence mask vector to mask the padded events.
            type_seq (tensor): [batch_size, seq_len], sequence of mark ids, with padded events having a mark of self.pad_token_id

        Returns:
            tuple: event loglike, non-event loglike, intensity at event with padding events masked
        """

        # First, add an epsilon to every marked intensity for stability
        lambda_at_event = lambda_at_event + self.eps
        lambdas_loss_samples = lambdas_loss_samples + self.eps

        # Compute components for each LL term
        log_marked_event_lambdas = lambda_at_event.log()
        log_total_event_lambdas = lambda_at_event.sum(dim=-1).log()
        total_sampled_lambdas = lambdas_loss_samples.sum(dim=-1)

        # Compute event LL - [batch_size, seq_len]
        event_ll = -F.nll_loss(
            log_marked_event_lambdas.permute(
                0, 2, 1
            ),  # mark dimension needs to come second, not third to match nll_loss specs
            target=type_seq,
            ignore_index=self.pad_token_id,  # Padded events have a pad_token_id as a value
            reduction="none",  # Does not aggregate, and replaces what would have been the log(marked intensity) with 0.
        )

        # Compute mark- & time-specific event LL
        time_ll_pos = torch.where(
            type_seq == self.pad_token_id, 0.0, log_total_event_lambdas
        )
        mark_ll = event_ll - time_ll_pos

        # Compute non-event LL [batch_size, seq_len]
        # interval_integral = length_interval * average of sampled lambda(t)
        if self.use_mc_samples:
            non_event_ll = (
                total_sampled_lambdas.mean(dim=-1) * time_delta_seq * seq_mask
            )
        else:  # Use trapezoid rule
            non_event_ll = (
                0.5
                * (
                    total_sampled_lambdas[..., 1:] + total_sampled_lambdas[..., :-1]
                ).mean(dim=-1)
                * time_delta_seq
                * seq_mask
            )

        num_events = torch.masked_select(event_ll, event_ll.ne(0.0)).size()[0]
        return event_ll, non_event_ll, num_events, mark_ll, time_ll_pos
    
    def predict_one_step_at_every_event(self, batch, **kwargs):
        """One-step prediction for every event in the sequence.
        We use the trapezoidal rule to estimate the _expected_ next event time,
        by re-using the configs under `thinning`. We do NOT use thinning sampling.
        Then mark is predicted conditioned on the est. expected next event time.

        Args:
            time_seqs (tensor): [batch_size, seq_len].
            time_delta_seqs (tensor): [batch_size, seq_len].
            type_seqs (tensor): [batch_size, seq_len].

        Returns:
            tuple: tensors of dtime and type prediction, [batch_size, seq_len].
        """
        time_seq, time_delta_seq, event_seq, batch_non_pad_mask, _ = batch

        # [batch_size, seq_len]
        n = self.event_sampler.num_sample
        e = self.event_sampler.num_exp

        # remove the last event, as the prediction based on the last event has no label
        # note: the first dts is 0
        # [batch_size, seq_len]
        time_seq, time_delta_seq, event_seq = (
            time_seq[:, :-1],
            time_delta_seq[:, :-1],
            event_seq[:, :-1],
        )

        us = (
            torch.linspace(0, math.pi / 2.0, n + 1)[:-1].to(self.device).split(e)
        )  # List of tensors of size (num_exp,)

        dtimes_pred = 0
        imp_lambda_int = 0
        last_u = None
        for u in us:
            # [0, 1, 2, 3, 4, 5] =split=> ([0, 1], [2, 3], [4, 5]) =concat=> ([0, 1], [1, 2, 3], [3, 4, 5])
            if last_u is not None:
                u = torch.concat([last_u[-1:], u])
            last_u = u

            u = u[None, None, ...].repeat(*time_seq.shape, 1)
            imp_samples = torch.tan(u)

            # (batch, seq_len, num_samples+1, num_marks)
            imp_marked_lambda = self.compute_intensities_at_sample_times(
                time_seq,  # (batch, seq_len)
                time_delta_seq,  # (batch, seq_len)
                event_seq,  # (batch, seq_len)
                imp_samples,  # (batch, seq_len, num_samples+1)
            )

            imp_lambda = imp_marked_lambda.sum(dim=-1)  # (batch, seq_len, num_samples)
            imp_lambda_int = imp_lambda_int + F.pad(
                torch.cumulative_trapezoid(imp_lambda, imp_samples),
                (1, 0),
                mode="constant",
                value=0.0,
            )  # (batch, seq_len, num_samples)

            # \int_0^\infty f(x) dx == \int_0^\infty t * p(t) dt == \int_0^\infty t * \lambda(t) * exp(-\int_0^t \lamda(s) ds) dt
            # \int_0^\infty f(x) dx == \int_0^(pi/2) f(tan u) / cos^2(u) du
            integrand = (
                imp_samples
                * imp_lambda
                * (-1 * imp_lambda_int).exp()
                / (torch.cos(u) ** 2)
            )

            dtimes_pred = dtimes_pred + torch.trapezoid(
                integrand, u
            )  # (batch, seq_len)
            imp_lambda_int = imp_lambda_int[:, :, -1:]


        intensities_at_times = self.compute_intensities_at_sample_times(time_seq,
                                                                        time_delta_seq,
                                                                        event_seq,
                                                                        dtimes_pred[..., None])  # (batch_size, seq_len, num_sample)

       
        intensities_normalized = intensities_at_times / intensities_at_times.sum(dim=-1, keepdim=True)

        # single sample
        intensities_weighted = intensities_normalized.squeeze(dim=-2)

        # [batch_size, seq_len]
        get_raw_mark_distribution = kwargs.get('get_raw_mark_distribution', False)
        if get_raw_mark_distribution:
            types_pred = intensities_weighted  # [batch_size, seq_len, num_marks]
        else:
            types_pred = torch.argmax(intensities_weighted, dim=-1)

        get_raw_pred_next_time = kwargs.get('get_raw_pred_next_time', False)
        if get_raw_pred_next_time:
            # [batch_size, seq_len, num_sample]
            dtimes_pred = dtimes_pred[..., None]
        return dtimes_pred, types_pred

#### S2P2 Wrapper

In [ ]:
class S2P2(TorchBaseModel):
    def __init__(self, model_config):
        super(S2P2, self).__init__()
        self.n_layers = model_config.num_layers
        self.P = model_config.model_specs["P"]
        self.H = model_config.model_specs["H"]
        self.beta = model_config.model_specs.get("beta", 1.0)
        self.bias = model_config.model_specs.get("bias", True)
        
        layer_kwargs = dict(
            P=self.P,
            H=self.H,
            dropout_rate=model_config.model_specs.get("dropout_rate", 0.0),
            pre_norm=model_config.model_specs.get("pre_norm", True),
            post_norm=model_config.model_specs.get("post_norm", False),
            relative_time=model_config.model_specs.get("relative_time", False)
        )
        
        self.layers = nn.ModuleList([Forward_LLH(**layer_kwargs, is_first_layer = i == 0) for i in range(self.layers)])
        
        # One embedding to share amongst layers to be used as input into a layer-specific and input-aware impulse
        self.layers_mark_emb = nn.Embedding(self.num_event_types_pad, self.H)
        
        self.intensity_net = self.IntensityNet(input_dim = self.H, 
                                               num_event_types = self.num_event_types, 
                                               bias_bool = self.bias)
        
    
    # assumes time has already been evolved, taking a vertical stack of hidden states and computing intensities    
    def _get_intensity(self, x_LP: Union[torch.tensor, List[torch.tensor]], right_us_BNH) -> torch.Tensor:
        left_u_H = None
        for i, layer in enumerate(self.layers):
            if isinstance (x_LP, list):
                left_u_H = layer.depth_pass(x_LP[i], current_left_u_H = left_u_H, prev_right_u_H = right_us_BNH[i])
            else:
                left_u_H = layer.depth_pass(x_LP[..., i, :], current_left_u_H = left_u_H, prev_right_u_H = right_us_BNH[i])
        return self.intensity_net(left_u_H) # calls IntensityNet's forward() by pytorch's nn.Module implementation of __call__
        
    def _evolve_and_get_intensity_at_sampled_dts(self, x_LP, dt_G, right_us_H):
        left_u_GH = None
        for i, layer in enumerate(self.layers):
            x_GP = layer.get_left_limit(
                right_limit_P = x_LP[..., i, :],
                dt_G = dt_G,
                next_left_u_GH = left_u_GH,
                current_right_u_H = right_us_H[i]
            )
            left_u_GH = layer.depth_pass(
                current_left_x_P = x_GP,
                current_left_u_H = left_u_GH,
                prev_right_u_H = right_us_H[i]
            ) 
        return self.intensity_net(left_u_GH)
    
    def forward(self, batch, x_0_BLP: Optional[torch.Tensor] = None, **kwargs) -> Tuple[torch.Tensor, torch.Tensor]:
        t_BN, dt_BN, marks_BN, batch_non_pad_mask, _ = batch # TODO: find out where batch structure lives
        
        right_xs_BNP = [] # list over layers: each is [B,N,P], including both t_0 and t_N
        left_xs_BNm1P = [] # list over layers: each is [B,N-1,P] (forward variant)
        right_us_BNH = [None] # list over layers+1: per layer right u, element 0 is None as this is the 'input' to the first layer
        left_u_BNH, right_u_BNH = None, None # current layer's input as layer depth is traversed, initialized to None for first layer
        α_BNH = self.layers_mark_emb(marks_BN) # this is the mark impulse embedding \alpha_k_i for each event in the batch
        
        for i, layer in enumerate(self.layers):
            x_0 = x_0_BLP[:, i] if x_0_BLP is not None else None
            x_BNP, next_layer_left_u_BNH, next_layer_right_u_BNH = layer.forward(left_u_BNH, right_u_BNH, α_BNH, dt_BN, x_0)
            assert next_layer_right_u_BNH is not None
            right_xs_BNP.append(x_BNP)
            
            # if NOT backward variant, compute left-limit x(t_i-) from right-limit x(t_{i-1}+)
            # here we are creating a list of left-limit xs at each event time except the first (t_1-, t_2-, ..., t_N-) using
            # the .get_left_limit() method of the layer, which evolves the right-limit state at t_{i-1}+ forward by dt_i 
            # to get left-limit state at t_i-.
            if next_layer_left_u_BNH is None: # NOT backward variant
                left_xs_BNm1P.append(
                    layer.get_left_limit(
                        x_BNP[..., :-1, :], # at time [t_0, t_1, ..., t_{N-1}]
                        dt_BN[..., 1:].unsqueeze(-1), # with dts [t1-t0, t2-t1, ..., t_N-t_{N-1}]
                        current_right_u_H = right_u_BNH if right_u_BNH is None else right_u_BNH[..., :-1, :], # at times [t_0, t_1, ..., t_{N-1}]
                        next_left_u_H = left_u_BNH if left_u_BNH is None else left_u_BNH[..., 1:, :].unsqueeze(-2) # at times [t_1, t_2 ..., t_N]
                    ).squeeze(-2) # get it back to shape [B,N-1,P] by removing the singleton dimension added in get_left_limit
                )
            right_us_BNH.append(next_layer_right_u_BNH)
            left_u_BNH, right_u_BNH = next_layer_left_u_BNH, next_layer_right_u_BNH
            
        right_xs_BNLP = torch.stack(right_xs_BNP, dim = -2)
        
        ret_val = {
            "right_xs_BNLP": right_xs_BNLP, # [t_0, ..., t_N]
            "right_us_BNH": right_us_BNH # [t_0, ..., t_N]; list starting with None
        }
        if left_u_BNH is not None: # backward variant
            ret_val["left_u_BNm1H"] = left_u_BNH[..., 1:, :]
        else: # NOT backward variant
            ret_val["left_xs_BNm1LP"] = torch.stack(left_xs_BNm1P, dim = -2)
            
        # 'seq_len - 1' left limit for [t_1, ..., t_N] for events (u if available, x if not)
        # 'seq_len' right limit for [t_0, t_1, ..., t_{N-1}, t_N] for events xs or us
        return ret_val
    
    def loglike_loss(self, batch, **kwargs):
        # This method is a sort of "orchestration" for the log likelihood specific to S2P2, which takes into account right/left limits,
        # whereas torch_basemodel.compute_loglikelihood is a more general MATHEMATICAL method that just computes the log-likelihood.
        # loglike_loss does the S2P2 forward pass, computes intensities at event times and sampled times, and the passes these agruments
        # into torch_basemodel.compute_loglikelihood to get the actual log-likelihood values for the S2P2 model.
        # hidden states at the left and right limits around event time; note for the shift by 1 in indices:
        # consider a sequence [t0, t1, ..., tN]
        # Produces the following:
        # left_x: x0, x1, x2, ... <-> x_{t_1-}, x_{t_2-}, x_{t_3-}, ..., x_{t_N-} (note the shift in indices) for all layers
        #    OR ==>               <-> u_{t_1-}, u_{t_2-}, u_{t_3-}, ..., u_{t_N-} for last layer
        #
        # right_x: x0, x1, x2, ... <-> x_{t_0+}, x_{t_1+}, ..., x_{t_N+} for all layers
        # right_u: u0, u1, u2, ... <-> u_{t_0+}, u_{t_1+}, ..., u_{t_N+} for all layers
        forward_results = self.forward(batch)  # N minus 1 comparing with sequence lengths
        right_xs_BNLP, right_us_BNH = (
            forward_results["right_xs_BNLP"],
            forward_results["right_us_BNH"],
        )
        right_us_BNm1H = [
            None if right_u_BNH is None else right_u_BNH[:, :-1, :]
            for right_u_BNH in right_us_BNH
        ]

        ts_BN, dts_BN, marks_BN, batch_non_pad_mask, _ = batch

        # evaluate intensity values at each event *from the left limit*, _get_intensity: [LP] -> [M]
        # left_xs_B_Nm1_LP = left_xs_BNm1LP[:, :-1, ...]  # discard the left limit of t_N
        # Note: no need to discard the left limit of t_N because "marks_mask" will deal with it
        if "left_u_BNm1H" in forward_results:  # ONLY backward variant
            intensity_B_Nm1_M = self.intensity_net(
                forward_results["left_u_BNm1H"]
            )  # self.ScaledSoftplus(self.linear(forward_results["left_u_BNm1H"]))
        else:  # NOT backward variant
            intensity_B_Nm1_M = self._get_intensity(
                forward_results["left_xs_BNm1LP"], right_us_BNm1H
            )

        # sample dt in each interval for MC: [batch_size, num_times=N-1, num_mc_sample]
        # N-1 because we only consider the intervals between N events
        # G for grid points
        dts_sample_B_Nm1_G = self.make_dtime_loss_samples(dts_BN[:, 1:])

        # evaluate intensity at dt_samples for MC *from the left limit* after decay -> shape (B, N-1, MC, M)
        intensity_dts_B_Nm1_G_M = self._evolve_and_get_intensity_at_sampled_dts(
            right_xs_BNLP[
                :, :-1
            ],  # x_{t_i+} will evolve up to x_{t_{i+1}-} and many times between for i=0,...,N-1
            dts_sample_B_Nm1_G,
            right_us_BNm1H,
        )

        event_ll, non_event_ll, num_events, mark_ll, time_ll_pos = (
            self.compute_loglikelihood(
                lambda_at_event=intensity_B_Nm1_M,
                lambdas_loss_samples=intensity_dts_B_Nm1_G_M,
                time_delta_seq=dts_BN[:, 1:],
                seq_mask=batch_non_pad_mask[:, 1:],
                type_seq=marks_BN[:, 1:],
            )
        )

        # compute extra statistics
        time_ll = time_ll_pos - non_event_ll

        # compute loss to optimize
        loss = -(event_ll - non_event_ll).sum()

        return_raw_ll = kwargs.get("return_raw_ll", False)
        res_dict = (
            {"non_event_ll": non_event_ll, "mark_intensity": intensity_B_Nm1_M}
            if return_raw_ll
            else None
        )

        return loss, num_events, mark_ll.sum(), time_ll.sum(), res_dict

    def compute_intensities_at_sample_times(
        self, event_times_BN, inter_event_times_BN, marks_BN, sample_dtimes, **kwargs
    ):
        """Compute the intensity at sampled times, not only event times.  *from the left limit*
        
        This method is a public-ish method used for prediction/simulation taht is passed into the thinning-based sampler:
        EventSampler.draw_next_time_one_step(). It's a higher-level wrapper whose job is essentially:
        "given the observed history (times & marks) and some propsed sample offsets, return intensities at those sample offsets"

        Args:
            time_seq (tensor): [batch_size, seq_len], times seqs.
            time_delta_seq (tensor): [batch_size, seq_len], time delta seqs.
            event_seq (tensor): [batch_size, seq_len], event type seqs.
            sample_dtimes (tensor): [batch_size, seq_len, num_sample], sampled inter-event timestamps.

        Returns:
            tensor: [batch_size, num_times, num_mc_sample, num_event_types],
                    intensity at each timestamp for each event type.
        """

        compute_last_step_only = kwargs.get("compute_last_step_only", False)

        # assume inter_event_times_BN always starts from 0
        _input = event_times_BN, inter_event_times_BN, marks_BN, None, None

        # 'seq_len - 1' left limit for [t_1, ..., t_N]
        # 'seq_len' right limit for [t_0, t_1, ..., t_{N-1}, t_N]

        forward_results = self.forward(
            _input
        )  # N minus 1 comparing with sequence lengths
        right_xs_BNLP, right_us_BNH = (
            forward_results["right_xs_BNLP"],
            forward_results["right_us_BNH"],
        )

        if (
            compute_last_step_only
        ):  # fix indices for right_us_BNH: list [None, tensor([BNH]), ...]
            right_us_B1H = [
                None if right_u_BNH is None else right_u_BNH[:, -1:, :]
                for right_u_BNH in right_us_BNH
            ]
            sampled_intensity = self._evolve_and_get_intensity_at_sampled_dts(
                right_xs_BNLP[:, -1:, :, :], sample_dtimes[:, -1:, :], right_us_B1H
            )  # equiv. to right_xs_BNLP[:, -1, :, :][:, None, ...]
        else:
            sampled_intensity = self._evolve_and_get_intensity_at_sampled_dts(
                right_xs_BNLP, sample_dtimes, right_us_BNH
            )
        return sampled_intensity  # [B, N, MC, M]

True